# exp_adaptive_gate — can a label-free signal route retrieval-vs-reranking per topic? (path E prototype)

The whole session's finding: TREC22 (detailed patients) rewards the reranking ensemble; TREC23 (sparse
questionnaires) rewards raw retrieval order. A **topic-adaptive gate** would pick per topic — output the
retrieval order for terse topics, the ensemble for detailed ones — letting one system win on both.

This prototype (CPU-only, cached data) tests the core question **before** any expensive rebuild:
does a **label-free** per-topic signal (length / specificity) predict *which ranker is better*?

Held-out sets only: TREC22 (50, ensemble helps) + TREC23 (37, retrieval helps). For each topic compute
retrieval-order NDCG@10 (rrf) and ensemble NDCG@10, the delta, and candidate gate signals. If a signal
correlates with the delta and a threshold captures most of the oracle-gate headroom, path E is real.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q datasets pytrec_eval

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np
from ctmatch.experiments import ExperimentConfig, load_eval, ndcg_at_k
DATA_ROOT = '/content/drive/MyDrive/ct_data23'; T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')

# ── TREC22 (held-out; ensemble helps here) ──
s22 = load_eval(cfg, ['trec22'])['trec22']; rel22 = s22['rel_dict']; t2t22 = s22['topic2text']
pool_nqs = json.load(open(cfg.path('data/pool_nqs.json'))); pool22 = pool_nqs['trec22']
rrf22 = {}
for l in open(cfg.path('data/retrieval_feats_nqs.jsonl')):
    r = json.loads(l)
    if r['source'] == 'trec22': rrf22[(r['topic_id'], r['doc_id'])] = r['rrf']
# ensemble per-topic NDCG@10 (cached by train_ensemble_full)
ens22 = {json.loads(l)['topic']: json.loads(l)['ndcg@10'] for l in open(cfg.path('results/per_topic_ndcg_nqs.jsonl'))}

# ── TREC23 (held-out; retrieval helps here) ──
topics23 = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel23 = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel23.setdefault(t, {})[d] = int(r)
topics23 = {t: x for t, x in topics23.items() if t in rel23}
pool23 = json.load(open(f'{T23}/pool_nqs_2023.json'))
rrf23 = {}
for l in open(f'{T23}/retrieval_feats_2023.jsonl'):
    r = json.loads(l); rrf23[(r['topic_id'], r['doc_id'])] = r['rrf']
# ensemble per-topic NDCG@10 from the saved run (run_ext2023.txt: topic Q0 doc rank score tag)
ens_run23 = {}
for l in open(f'{T23}/run_ext2023.txt'):
    t, _, d, rk, sc, _tag = l.split(); ens_run23.setdefault(t, {})[d] = float(sc)
print('TREC22 topics', len(rel22), '| TREC23 topics', len(topics23))

In [ ]:
# Per-topic: retrieval-order NDCG, ensemble NDCG, delta = ret - ens (positive => route to RETRIEVAL),
# and cheap label-free signals.
def signals(text):
    words = text.split()
    return {'len': len(words),
            'ndig': sum(c.isdigit() for c in text),
            'nsent': text.count('.') + text.count(';') + 1,
            'digfrac': (sum(c.isdigit() for c in text) / max(len(text), 1))}

rows = []
# TREC22
for t in rel22:
    if t not in t2t22 or t not in ens22: continue
    ret = ndcg_at_k(sorted(pool22[t], key=lambda d: rrf22.get((t, d), -1e9), reverse=True), rel22[t])
    rows.append({'ds': 'TREC22', 't': t, 'ret': ret, 'ens': ens22[t], **signals(t2t22[t])})
# TREC23
for t in topics23:
    ret = ndcg_at_k(sorted(pool23[t], key=lambda d: rrf23.get((t, d), -1e9), reverse=True), rel23[t])
    ens = ndcg_at_k(sorted(ens_run23[t], key=lambda d: ens_run23[t][d], reverse=True), rel23[t])
    rows.append({'ds': 'TREC23', 't': t, 'ret': ret, 'ens': ens, **signals(topics23[t])})
for r in rows: r['delta'] = r['ret'] - r['ens']

for ds in ['TREC22', 'TREC23']:
    R = [r for r in rows if r['ds'] == ds]
    print(f"{ds}: n={len(R)}  ret={np.mean([r['ret'] for r in R]):.3f}  ens={np.mean([r['ens'] for r in R]):.3f}  "
          f"mean len={np.mean([r['len'] for r in R]):.0f}  ndig={np.mean([r['ndig'] for r in R]):.0f}  "
          f"prefer-retrieval={np.mean([r['delta']>0 for r in R]):.0%}")

In [ ]:
# Does a label-free signal predict the delta (which ranker is better)? And can a threshold route well?
allrows = rows
d = np.array([r['delta'] for r in allrows])
print('correlation(signal, delta)  [delta>0 => retrieval better; want a signal that separates]')
for s in ['len', 'ndig', 'nsent', 'digfrac']:
    x = np.array([r[s] for r in allrows]); c = np.corrcoef(x, d)[0, 1]
    print(f'  {s:8s} r = {c:+.3f}')

def gated_mean(route_ret):   # route_ret[i] True => use retrieval order for topic i
    per_ds = {}
    for ds in ['TREC22', 'TREC23']:
        vals = [(r['ret'] if route_ret[i] else r['ens']) for i, r in enumerate(allrows) if r['ds'] == ds]
        per_ds[ds] = float(np.mean(vals))
    return per_ds

always_ens = gated_mean([False]*len(allrows))
always_ret = gated_mean([True]*len(allrows))
oracle     = gated_mean([r['delta'] > 0 for r in allrows])       # perfect per-topic routing (ceiling)
print('\n                     TREC22     TREC23')
print(f"always-ensemble   {always_ens['TREC22']:.4f}   {always_ens['TREC23']:.4f}   (current behavior)")
print(f"always-retrieval  {always_ret['TREC22']:.4f}   {always_ret['TREC23']:.4f}")
print(f"ORACLE gate       {oracle['TREC22']:.4f}   {oracle['TREC23']:.4f}   (ceiling of perfect routing)")

# Best single-signal threshold gate (route to retrieval below/above a cut). Report the best signal+cut
# and its per-dataset result. NOTE: cut chosen on this data — a real gate sets it a-priori; this measures
# the *achievable* headroom of the signal, read alongside the ORACLE ceiling.
best = None
for s in ['len', 'ndig', 'nsent', 'digfrac']:
    xs = sorted({r[s] for r in allrows})
    for cut in xs:
        for direction in ['below', 'above']:
            route = [(r[s] < cut) if direction == 'below' else (r[s] >= cut) for r in allrows]
            g = gated_mean(route)
            combined = g['TREC22'] + g['TREC23']   # want both high
            if best is None or combined > best[0]:
                best = (combined, s, cut, direction, g)
_, s, cut, direction, g = best
print(f"\nbest threshold gate: route to RETRIEVAL when {s} {'<' if direction=='below' else '>='} {cut}")
print(f"                  {g['TREC22']:.4f}   {g['TREC23']:.4f}")
print('\nVERDICT: if ORACLE and the threshold gate both keep TREC22 near 0.575 AND lift TREC23 toward the')
print('retrieval level (~0.47), one system wins on both -> path E is real, proceed to the full rebuild.')